# 🏗️ Notebook 8: Complete Transformer Block

**Assembling All Components Together**

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Understand **residual connections** (skip connections)
2. Implement **layer normalization**
3. Combine **attention + feed-forward**
4. Build a complete **Transformer Block**
5. Stack multiple blocks into a **GPT model**
6. Understand the **complete architecture**

---

## 📚 Table of Contents

1. [Residual Connections](#1-residual-connections)
2. [Layer Normalization](#2-layer-normalization)
3. [Complete Transformer Block](#3-complete-transformer-block)
4. [Stacking Blocks](#4-stacking-blocks)
5. [Complete GPT Model](#5-complete-gpt-model)
6. [Architecture Visualization](#6-architecture-visualization)
7. [Key Takeaways](#7-key-takeaways)

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import os

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

### Environment Setup (Colab/Local)

This cell detects whether you're running in Google Colab or locally and sets up the environment accordingly.

In [ ]:
# ========================================
# ENVIRONMENT DETECTION & SETUP
# ========================================

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

# Setup directories
if IN_COLAB:
    # Create necessary directories for Colab
    os.makedirs('data', exist_ok=True)
    os.makedirs('visualizations/tokenization', exist_ok=True)
    os.makedirs('visualizations/embeddings', exist_ok=True)
    os.makedirs('visualizations/data_pipeline', exist_ok=True)
    print("Created directories")
    
    # Set paths for Colab
    DATA_DIR = 'data'
    VIZ_DIR = 'visualizations'
else:
    # Use relative paths for local execution
    DATA_DIR = '../data'
    VIZ_DIR = '../visualizations'
    
    # Create directories if they don't exist (LOCAL FIX)
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/tokenization', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/embeddings', exist_ok=True)
    os.makedirs(f'{VIZ_DIR}/data_pipeline', exist_ok=True)
    print(f"Created directories: {DATA_DIR}, {VIZ_DIR}")

print(f"\nData directory: {DATA_DIR}")
print(f"Visualization directory: {VIZ_DIR}")

In [ ]:
# Load configuration
with open(f'{DATA_DIR}/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

vocab_size = config['vocab_size']
block_size = config['block_size']
batch_size = config['batch_size']

# Load embedding config
checkpoint = torch.load(f'{DATA_DIR}/embedding_layer.pth')
n_embd = checkpoint['n_embd']

print(f"📊 Configuration:")
print(f"Vocabulary size: {vocab_size}")
print(f"Block size:      {block_size}")
print(f"Embedding dim:   {n_embd}")

---

## 1. Residual Connections

### 🚨 The Vanishing Gradient Problem

In deep networks:
```
Layer 1 → Layer 2 → ... → Layer 12
```

**Problem**: Gradients become very small as they backpropagate through many layers.

### 💡 The Solution: Residual Connections

Instead of:
```
x → Layer → output
```

We use:
```
x → Layer → output + x  (add input back!)
```

**Benefits:**
- Gradients can flow directly through skip connections
- Easier to train deep networks
- Model learns "refinements" rather than complete transformations

In [ ]:
# Simple example of residual connection
x = torch.randn(4, 8, n_embd)
layer = nn.Linear(n_embd, n_embd)

# Without residual
output_no_residual = layer(x)

# With residual
output_with_residual = layer(x) + x  # Add input back!

print(f"📊 Input shape:              {x.shape}")
print(f"📤 Output (no residual):     {output_no_residual.shape}")
print(f"📤 Output (with residual):   {output_with_residual.shape}")
print(f"\n💡 Residual connection: output = f(x) + x")

---

## 2. Layer Normalization

### 🎯 Why Normalize?

During training, activations can have varying scales:
- Some features might be very large
- Others might be very small
- This makes training unstable

### 💡 Layer Normalization

Normalize across the feature dimension:

```
For each token:
  mean = average of all features
  std = standard deviation of all features
  normalized = (x - mean) / std
  output = γ * normalized + β  (learnable scale & shift)
```

In [ ]:
# Create layer normalization
ln = nn.LayerNorm(n_embd)

# Test data with varying scales
x = torch.randn(4, 8, n_embd) * 10  # Large scale

# Apply layer norm
x_normalized = ln(x)

print(f"📊 Before LayerNorm:")
print(f"   Mean: {x.mean():.4f}")
print(f"   Std:  {x.std():.4f}")
print(f"   Min:  {x.min():.4f}")
print(f"   Max:  {x.max():.4f}")

print(f"\n📊 After LayerNorm:")
print(f"   Mean: {x_normalized.mean():.4f}")
print(f"   Std:  {x_normalized.std():.4f}")
print(f"   Min:  {x_normalized.min():.4f}")
print(f"   Max:  {x_normalized.max():.4f}")

print(f"\n💡 LayerNorm stabilizes activations!")

### 📊 Visualize Normalization Effect

In [ ]:
# Visualize before and after normalization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
axes[0].hist(x.flatten().numpy(), bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Value', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0].set_title('Before LayerNorm\n(Wide distribution)', fontsize=13, fontweight='bold')
axes[0].axvline(x.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {x.mean():.2f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# After
axes[1].hist(x_normalized.flatten().detach().numpy(), bins=50, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_xlabel('Value', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[1].set_title('After LayerNorm\n(Normalized distribution)', fontsize=13, fontweight='bold')
axes[1].axvline(x_normalized.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {x_normalized.mean():.2f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/layer_norm_effect.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 LayerNorm centers and scales the distribution!")

---

## 3. Complete Transformer Block

### 🏗️ The Architecture

```
Input
  ↓
LayerNorm → Multi-Head Attention → Add (residual)
  ↓
LayerNorm → Feed-Forward → Add (residual)
  ↓
Output
```

**Key Pattern**: LayerNorm → Sublayer → Residual

### 💻 Load Previous Components

In [ ]:
# Import components from previous notebooks

class SelfAttentionHead(nn.Module):
    """One head of self-attention."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.head_size = head_size
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) / (self.head_size ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""
    def __init__(self, n_embd, num_heads, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % num_heads == 0
        self.heads = nn.ModuleList([
            SelfAttentionHead(n_embd, n_embd // num_heads, block_size, dropout)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    """Position-wise feed-forward network."""
    def __init__(self, n_embd, expansion_factor=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd * expansion_factor),
            nn.GELU(),
            nn.Linear(n_embd * expansion_factor, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        return self.net(x)

print("✅ All components loaded!")

### 🏗️ Build Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    """
    One Transformer block: communication (attention) + computation (FFN).
    
    Args:
        n_embd: Embedding dimension
        num_heads: Number of attention heads
        block_size: Maximum sequence length
        dropout: Dropout probability
    """
    
    def __init__(self, n_embd, num_heads, block_size, dropout=0.1):
        super().__init__()
        
        # Multi-head attention
        self.sa = MultiHeadAttention(n_embd, num_heads, block_size, dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(n_embd, expansion_factor=4, dropout=dropout)
        
        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        """
        Args:
            x: Input, shape (B, T, C)
        
        Returns:
            out: Output, shape (B, T, C)
        """
        # Attention block with residual
        x = x + self.sa(self.ln1(x))  # Pre-norm + residual
        
        # Feed-forward block with residual
        x = x + self.ffn(self.ln2(x))  # Pre-norm + residual
        
        return x

# Create a transformer block
num_heads = 4
block = TransformerBlock(n_embd, num_heads, block_size)

print(f"🏗️ Transformer Block Created!")
print(f"\n📊 Configuration:")
print(f"   Embedding dim: {n_embd}")
print(f"   Num heads:     {num_heads}")
print(f"   Block size:    {block_size}")

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
print(f"\n💾 Total parameters: {total_params:,}")

In [ ]:
# Test the transformer block
test_input = torch.randn(4, 8, n_embd)
test_output = block(test_input)

print(f"📥 Input shape:  {test_input.shape}")
print(f"📤 Output shape: {test_output.shape}")
print(f"\n✅ Transformer block preserves shape: (B, T, C) → (B, T, C)")

---

## 4. Stacking Blocks

### 🎯 Deep Networks

GPT models stack multiple Transformer blocks:

```
Input Embeddings
  ↓
Transformer Block 1
  ↓
Transformer Block 2
  ↓
...
  ↓
Transformer Block N
  ↓
Output
```

**Examples:**
- GPT-2 Small: 12 blocks
- GPT-2 Medium: 24 blocks
- GPT-3: 96 blocks!

In [ ]:
# Stack multiple blocks
n_layer = 6  # Number of transformer blocks

blocks = nn.Sequential(*[
    TransformerBlock(n_embd, num_heads, block_size)
    for _ in range(n_layer)
])

print(f"🏗️ Stacked {n_layer} Transformer Blocks!")

# Count total parameters
total_params = sum(p.numel() for p in blocks.parameters())
print(f"\n💾 Total parameters: {total_params:,}")

# Test
test_input = torch.randn(4, 8, n_embd)
test_output = blocks(test_input)

print(f"\n📥 Input shape:  {test_input.shape}")
print(f"📤 Output shape: {test_output.shape}")

---

## 5. Complete GPT Model

### 🎯 Full Architecture

```
Token IDs
  ↓
Token Embeddings + Positional Embeddings
  ↓
Transformer Block 1
  ↓
Transformer Block 2
  ↓
...
  ↓
Transformer Block N
  ↓
Layer Norm
  ↓
Linear (to vocab size)
  ↓
Logits (predictions)
```

In [ ]:
class GPTLanguageModel(nn.Module):
    """
    Complete GPT Language Model.
    
    Args:
        vocab_size: Size of vocabulary
        n_embd: Embedding dimension
        block_size: Maximum sequence length
        n_layer: Number of transformer blocks
        num_heads: Number of attention heads
        dropout: Dropout probability
    """
    
    def __init__(self, vocab_size, n_embd, block_size, n_layer, num_heads, dropout=0.1):
        super().__init__()
        
        # Token and position embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        # Transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embd, num_heads, block_size, dropout)
            for _ in range(n_layer)
        ])
        
        # Final layer norm
        self.ln_f = nn.LayerNorm(n_embd)
        
        # Language modeling head
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        self.block_size = block_size
    
    def forward(self, idx, targets=None):
        """
        Args:
            idx: Token indices, shape (B, T)
            targets: Target indices for loss calculation, shape (B, T)
        
        Returns:
            logits: Predictions, shape (B, T, vocab_size)
            loss: Cross-entropy loss (if targets provided)
        """
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_embedding_table(idx)  # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C)
        
        # Transformer blocks
        x = self.blocks(x)  # (B, T, C)
        
        # Final layer norm
        x = self.ln_f(x)  # (B, T, C)
        
        # Language modeling head
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        # Calculate loss if targets provided
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        """
        Generate new tokens.
        
        Args:
            idx: Starting indices, shape (B, T)
            max_new_tokens: Number of tokens to generate
        
        Returns:
            idx: Extended sequence, shape (B, T + max_new_tokens)
        """
        for _ in range(max_new_tokens):
            # Crop to block_size
            idx_cond = idx[:, -self.block_size:]
            
            # Get predictions
            logits, _ = self(idx_cond)
            
            # Focus on last time step
            logits = logits[:, -1, :]  # (B, C)
            
            # Apply softmax
            probs = F.softmax(logits, dim=-1)  # (B, C)
            
            # Sample
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            # Append
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        
        return idx

# Create GPT model
model = GPTLanguageModel(
    vocab_size=vocab_size,
    n_embd=n_embd,
    block_size=block_size,
    n_layer=6,
    num_heads=4,
    dropout=0.1
)

print(f"🎉 Complete GPT Model Created!")
print(f"\n📊 Model Configuration:")
print(f"   Vocabulary size: {vocab_size}")
print(f"   Embedding dim:   {n_embd}")
print(f"   Block size:      {block_size}")
print(f"   Num layers:      6")
print(f"   Num heads:       4")

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n💾 Total parameters: {total_params:,}")
print(f"\n💡 For comparison:")
print(f"   GPT-2 Small:  124M parameters")
print(f"   GPT-3:        175B parameters")
print(f"   Our model:    {total_params:,} parameters")

In [ ]:
# Test the model
test_idx = torch.randint(0, vocab_size, (4, 8))
test_targets = torch.randint(0, vocab_size, (4, 8))

logits, loss = model(test_idx, test_targets)

print(f"📥 Input shape:   {test_idx.shape}")
print(f"📤 Logits shape:  {logits.shape}")
print(f"📉 Loss:          {loss.item():.4f}")
print(f"\n✅ Model is ready for training!")

---

## 6. Architecture Visualization

### 📊 Parameter Distribution

In [ ]:
# Analyze parameter distribution
component_params = {
    'Token Embeddings': sum(p.numel() for p in model.token_embedding_table.parameters()),
    'Position Embeddings': sum(p.numel() for p in model.position_embedding_table.parameters()),
    'Transformer Blocks': sum(p.numel() for p in model.blocks.parameters()),
    'Final LayerNorm': sum(p.numel() for p in model.ln_f.parameters()),
    'LM Head': sum(p.numel() for p in model.lm_head.parameters())
}

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

components = list(component_params.keys())
params = list(component_params.values())
colors = ['steelblue', 'coral', 'green', 'purple', 'orange']

bars = ax.barh(components, params, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

# Add value labels
for bar, param in zip(bars, params):
    width = bar.get_width()
    percentage = (param / total_params) * 100
    ax.text(width, bar.get_y() + bar.get_height()/2,
           f'{param:,} ({percentage:.1f}%)',
           ha='left', va='center', fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Number of Parameters', fontsize=12, fontweight='bold')
ax.set_title('GPT Model Parameter Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/embeddings/parameter_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Most parameters are in Transformer blocks!")

---

## 7. Key Takeaways

### ✅ What We Learned

1. **Residual Connections**
   - Add input back to output: `output = f(x) + x`
   - Enables gradient flow in deep networks
   - Model learns refinements, not complete transformations

2. **Layer Normalization**
   - Normalizes across feature dimension
   - Stabilizes training
   - Applied before each sublayer (pre-norm)

3. **Transformer Block**
   ```
   x = x + Attention(LayerNorm(x))
   x = x + FFN(LayerNorm(x))
   ```
   - Two sublayers: attention + feed-forward
   - Each with pre-norm and residual

4. **Complete GPT Architecture**
   - Embeddings (token + position)
   - Stack of N transformer blocks
   - Final layer norm
   - Language modeling head

5. **Parameter Distribution**
   - ~70% in transformer blocks
   - ~15% in embeddings
   - ~15% in LM head

### 🎯 Key Insights

- **Residuals are critical**: Enable training of deep networks
- **Pre-norm is standard**: LayerNorm before sublayers
- **Scalability**: Just stack more blocks for larger models
- **Shape preservation**: (B, T, C) throughout the network

### 🔮 What's Next?

In **Notebook 9**, we'll implement **Training & Text Generation**:
- Training loop with optimization
- Loss calculation and monitoring
- Text generation strategies
- Temperature, top-k, top-p sampling
- Complete end-to-end example!

In [ ]:
# Save the complete model
torch.save({
    'model': model.state_dict(),
    'vocab_size': vocab_size,
    'n_embd': n_embd,
    'block_size': block_size,
    'n_layer': 6,
    'num_heads': 4
}, f'{DATA_DIR}/gpt_model.pth')

print("✅ Complete GPT model saved to f'{DATA_DIR}/gpt_model.pth'")
print("\n🎉 You've built a complete Transformer from scratch!")